
# 02 - Camada Silver | Tratamento e enriquecimento
## Projeto: Pipeline de Imóveis BH
## 
## Objetivo: Tratar valores ausentes, adicionar novas colunas
### 
### AUTOR : Eduardo Cosme Pereira dos santos

In [0]:
## Importar pacotes utilizados
import pandas as pd
import numpy as np
from pyspark.sql.types import *
from pyspark.sql.functions import *
from pyspark.sql import functions as F
from itertools import chain
import logging


logging.info("Pacotes importados com sucesso !!")

In [0]:
## Configuração do logging - logs do projeto 


logging.basicConfig(
    level=logging.INFO, filename="workspace/zap_imoveis/bronze.log",
    format="%(asctime)s - %(levelname)s - %(message)s"
)

In [0]:
## Carregar dados da camada bronze
path_bronze = "/Volumes/workspace/zap_imoveis/bronze"

zap_silver = spark.read.format("delta").load(f"{path_bronze}/zap_imoveis")

logging.info("Dados da camada bronze carregados com sucesso !!")


## Visualizar dados
display(zap_silver.head(10))

# Verificar os tipos de dados no dataset  
zap_silver.printSchema()

preco,condominio,area,quartos,bairro,latitude,longitude,tipo,banheiros,vagas,iptu,data_atualizacao
6500,1535,680,3,Cruzeiro,null,null,APARTMENT,3,2.0,555,2026-05-12T22:52:01.415Z
2500,300,98,3,Milionários (Barreiro),null,null,APARTMENT,2,1.0,198,2026-05-12T22:52:01.415Z
2800,0,120,3,Ipê,null,null,HOME,2,1.0,100,2026-05-12T22:52:01.415Z
4700,512,1000,3,Buritis,null,null,APARTMENT,3,2.0,412,2026-05-12T22:52:01.415Z
4000,630,80,3,Buritis,-19.971905,-43.965046,APARTMENT,2,2.0,2554,2026-05-12T22:52:01.415Z
3500,350,720,3,Silveira,null,null,APARTMENT,3,3.0,266,2026-05-12T22:52:01.415Z
3300,550,90,3,Estrela do Oriente,-19.964745,-43.984511,APARTMENT,2,2.0,290,2026-05-12T22:52:01.415Z
6000,0,180,3,Paquetá,null,null,HOME,2,2.0,155,2026-05-12T22:52:01.415Z
3000,380,116,3,Caiçara-Adelaide,null,null,APARTMENT,2,2.0,190,2026-05-12T22:52:01.415Z
1950,null,null,3,Conjunto Paulo VI,-19.83209,-43.88831,HOME,2,2.0,0,2026-05-12T22:52:01.415Z


root
 |-- preco: integer (nullable = true)
 |-- condominio: integer (nullable = true)
 |-- area: integer (nullable = true)
 |-- quartos: integer (nullable = true)
 |-- bairro: string (nullable = true)
 |-- latitude: double (nullable = true)
 |-- longitude: double (nullable = true)
 |-- tipo: string (nullable = true)
 |-- banheiros: integer (nullable = true)
 |-- vagas: double (nullable = true)
 |-- iptu: integer (nullable = true)
 |-- data_atualizacao: timestamp (nullable = true)



In [0]:
## Verificar total de linhas e colunas do dataset
print(f"Total de linhas: {zap_silver.count()}")
print(f"Total de colunas: {len(zap_silver.columns)}")

Total de linhas: 1050
Total de colunas: 12


In [0]:
## Padronizar todos os nomes das colunas para maiusculas
zap_silver = zap_silver.toDF(*map(str.upper, zap_silver.columns))

## Visualizar dados
display(zap_silver.head(10))



PRECO,CONDOMINIO,AREA,QUARTOS,BAIRRO,LATITUDE,LONGITUDE,TIPO,BANHEIROS,VAGAS,IPTU,DATA_ATUALIZACAO
6500,1535,680,3,Cruzeiro,null,null,APARTMENT,3,2.0,555,2026-05-12T22:52:01.415Z
2500,300,98,3,Milionários (Barreiro),null,null,APARTMENT,2,1.0,198,2026-05-12T22:52:01.415Z
2800,0,120,3,Ipê,null,null,HOME,2,1.0,100,2026-05-12T22:52:01.415Z
4700,512,1000,3,Buritis,null,null,APARTMENT,3,2.0,412,2026-05-12T22:52:01.415Z
4000,630,80,3,Buritis,-19.971905,-43.965046,APARTMENT,2,2.0,2554,2026-05-12T22:52:01.415Z
3500,350,720,3,Silveira,null,null,APARTMENT,3,3.0,266,2026-05-12T22:52:01.415Z
3300,550,90,3,Estrela do Oriente,-19.964745,-43.984511,APARTMENT,2,2.0,290,2026-05-12T22:52:01.415Z
6000,0,180,3,Paquetá,null,null,HOME,2,2.0,155,2026-05-12T22:52:01.415Z
3000,380,116,3,Caiçara-Adelaide,null,null,APARTMENT,2,2.0,190,2026-05-12T22:52:01.415Z
1950,null,null,3,Conjunto Paulo VI,-19.83209,-43.88831,HOME,2,2.0,0,2026-05-12T22:52:01.415Z


In [0]:
## importar a metrica de dados nulos gerada na camada bronze
df_metricas =  spark.read.format("delta").load(f"{path_bronze}/metricas_bronze")

## Visualizar dados
display(df_metricas)


preco,condominio,area,quartos,bairro,latitude,longitude,tipo,banheiros,vagas,iptu
0.0,2.86,5.81,0.0,0.0,55.14,55.14,0.0,0.0,1.52,4.67


In [0]:
## Verificar o tipos de nomes na coluna tipo 

zap_silver.groupBy("tipo").count().show()


## Verificar o tipos de nomes na coluna bairro 

zap_silver.groupBy("bairro").count().show()

+--------------------+-----+
|                tipo|count|
+--------------------+-----+
|                HOME|  125|
|              OFFICE|    7|
|           APARTMENT|  778|
|           PENTHOUSE|  119|
| COMMERCIAL_PROPERTY|   12|
| COMMERCIAL_BUILDING|    1|
|SHED_DEPOSIT_WARE...|    1|
|         CONDOMINIUM|    4|
|            BUSINESS|    3|
+--------------------+-----+

+--------------------+-----+
|              bairro|count|
+--------------------+-----+
|Lagoinha Leblon (...|    2|
|       Santo Antônio|   18|
|          Califórnia|    1|
|  Cardoso (Barreiro)|    2|
|      Alípio de Melo|    7|
|       Frei Leopoldo|    1|
|            Araguaia|    1|
|       São Francisco|    1|
|             Paquetá|    9|
|    Caiçara-Adelaide|    3|
|     Padre Eustáquio|   23|
|           Jaqueline|    1|
|    Tirol (Barreiro)|    2|
|             Betânia|    2|
|         Vila Cloris|    6|
|             Ventosa|    2|
|Serra Verde (Vend...|    1|
|            Comiteco|    1|
|           S

In [0]:
## Padronizar os nomes da coluna tipo
zap_silver = zap_silver.withColumn("tipo", upper(trim(col("tipo"))))


## Criar um mapeamento dos tipos de imoveis gerar uma nova coluna
mapeamento = {
    "HOME": "CASA",
    "APARTMENT": "APARTAMENTO",
    "OFFICE": "ESCRITORIO",
    "PENTHOUSE": "COBERTURA",
    "COMMERCIAL_PROPERTY": "IMOVEL COMERCIAL",
    "COMMERCIAL_BUILDING": "PREDIO COMERCIAL",
    "SHED_DEPOSIT_WAREHOUSE ": "GALPÃO",
    "CONDOMINIUM": "CONDOMINIO",
    "BUSINESS": "EMPRESA",
}

## aplicar o mapeamento dentro de uma lista 
mapeamento_lista = create_map([lit(x) for x in chain(*mapeamento.items())])

## Criar uma nova coluna com os valores mapeados
zap_silver = zap_silver.withColumn("TIPO_DE_IMOVEL", mapeamento_lista[col("TIPO")])

logging.info("coluna tipo de imovel criada com sucesso ! ")


## Padronizar os nomes da coluna bairro
zap_silver = zap_silver.withColumn("BAIRRO", upper(trim(col("BAIRRO"))))


##visuarizar os dados
display(zap_silver.head(10))




PRECO,CONDOMINIO,AREA,QUARTOS,BAIRRO,LATITUDE,LONGITUDE,tipo,BANHEIROS,VAGAS,IPTU,DATA_ATUALIZACAO,TIPO DE IMOVEL,TIPO_DE_IMOVEL
6500,1535,680,3,CRUZEIRO,null,null,APARTMENT,3,2.0,555,2026-05-12T22:52:01.415Z,APARTAMENTO,APARTAMENTO
2500,300,98,3,MILIONÁRIOS (BARREIRO),null,null,APARTMENT,2,1.0,198,2026-05-12T22:52:01.415Z,APARTAMENTO,APARTAMENTO
2800,0,120,3,IPÊ,null,null,HOME,2,1.0,100,2026-05-12T22:52:01.415Z,CASA,CASA
4700,512,1000,3,BURITIS,null,null,APARTMENT,3,2.0,412,2026-05-12T22:52:01.415Z,APARTAMENTO,APARTAMENTO
4000,630,80,3,BURITIS,-19.971905,-43.965046,APARTMENT,2,2.0,2554,2026-05-12T22:52:01.415Z,APARTAMENTO,APARTAMENTO
3500,350,720,3,SILVEIRA,null,null,APARTMENT,3,3.0,266,2026-05-12T22:52:01.415Z,APARTAMENTO,APARTAMENTO
3300,550,90,3,ESTRELA DO ORIENTE,-19.964745,-43.984511,APARTMENT,2,2.0,290,2026-05-12T22:52:01.415Z,APARTAMENTO,APARTAMENTO
6000,0,180,3,PAQUETÁ,null,null,HOME,2,2.0,155,2026-05-12T22:52:01.415Z,CASA,CASA
3000,380,116,3,CAIÇARA-ADELAIDE,null,null,APARTMENT,2,2.0,190,2026-05-12T22:52:01.415Z,APARTAMENTO,APARTAMENTO
1950,null,null,3,CONJUNTO PAULO VI,-19.83209,-43.88831,HOME,2,2.0,0,2026-05-12T22:52:01.415Z,CASA,CASA


In [0]:
## Criar a coluna Segmento para a partir da coluna tipo de imovel 

## Categorizar 
residencial = ["CASA", "APARTAMENTO", "COBERTURA"]
comercial = ["ESCRITORIO", "PREDIO COMERCIAL", "GALPÃO", "IMOVEL COMERCIAL", "EMPRESA", "CONDOMINIO"]

zap_silver = zap_silver.withColumn("SEGMENTO", when(col("TIPO_DE_IMOVEL").isin(residencial), "RESIDENCIAL").otherwise("COMERCIAL"))

logging.info("coluna segmento criada com sucesso ! ")

## Fazer a contagem dos segmentos e confrontar com o total de registros
zap_silver.groupBy("SEGMENTO").count().show()
display(zap_silver.head(10))


+-----------+-----+
|   SEGMENTO|count|
+-----------+-----+
|RESIDENCIAL| 1022|
|  COMERCIAL|   28|
+-----------+-----+



PRECO,CONDOMINIO,AREA,QUARTOS,BAIRRO,LATITUDE,LONGITUDE,tipo,BANHEIROS,VAGAS,IPTU,DATA_ATUALIZACAO,TIPO DE IMOVEL,TIPO_DE_IMOVEL,SEGMENTO
6500,1535,680,3,CRUZEIRO,null,null,APARTMENT,3,2.0,555,2026-05-12T22:52:01.415Z,APARTAMENTO,APARTAMENTO,RESIDENCIAL
2500,300,98,3,MILIONÁRIOS (BARREIRO),null,null,APARTMENT,2,1.0,198,2026-05-12T22:52:01.415Z,APARTAMENTO,APARTAMENTO,RESIDENCIAL
2800,0,120,3,IPÊ,null,null,HOME,2,1.0,100,2026-05-12T22:52:01.415Z,CASA,CASA,RESIDENCIAL
4700,512,1000,3,BURITIS,null,null,APARTMENT,3,2.0,412,2026-05-12T22:52:01.415Z,APARTAMENTO,APARTAMENTO,RESIDENCIAL
4000,630,80,3,BURITIS,-19.971905,-43.965046,APARTMENT,2,2.0,2554,2026-05-12T22:52:01.415Z,APARTAMENTO,APARTAMENTO,RESIDENCIAL
3500,350,720,3,SILVEIRA,null,null,APARTMENT,3,3.0,266,2026-05-12T22:52:01.415Z,APARTAMENTO,APARTAMENTO,RESIDENCIAL
3300,550,90,3,ESTRELA DO ORIENTE,-19.964745,-43.984511,APARTMENT,2,2.0,290,2026-05-12T22:52:01.415Z,APARTAMENTO,APARTAMENTO,RESIDENCIAL
6000,0,180,3,PAQUETÁ,null,null,HOME,2,2.0,155,2026-05-12T22:52:01.415Z,CASA,CASA,RESIDENCIAL
3000,380,116,3,CAIÇARA-ADELAIDE,null,null,APARTMENT,2,2.0,190,2026-05-12T22:52:01.415Z,APARTAMENTO,APARTAMENTO,RESIDENCIAL
1950,null,null,3,CONJUNTO PAULO VI,-19.83209,-43.88831,HOME,2,2.0,0,2026-05-12T22:52:01.415Z,CASA,CASA,RESIDENCIAL


In [0]:
## Criar a coluna região para categorizar os bairros 

## Categorizar 
CENTRO_SUL  = ["SAVASSI", "LIBERDADE", "ANCHIETA", "FUNCIONÁRIOS", "SANTO AGOSTINHO", "LOURDES", "CARMO", "BELVEDERE", "SÃO PEDRO", "BOA VIAGEM", "SAGRADA FAMÍLIA", "SÃO LUIZ", "PADRE EUSTÁQUIO", "SERRA", "CENTRO", "SION", "SANTO ANTÔNIO", "BARRO PRETO", "SANTA EFIGÊNIA", "CRUZEIRO", "SÃO PAULO", "PRADO", "OURO PRETO", "FLORESTA", "GUTIERREZ", "LAGOINHA", "BOA VISTA", "APARECIDA", "SANTA LÚCIA", "POMPÉIA", "CIDADE NOVA", "NOVA SUÍSSA", "GLÓRIA", "SÃO GABRIEL", "IPÊ", "CORAÇÃO EUCARÍSTICO", "SÃO SALVADOR", "SANTA MARIA", "JARDIM MONTANHÊS", "SÃO JOSÉ", "SANTA TEREZA", "SANTA AMÉLIA", "ARAGUAIA", "ALTO DOS PINHEIROS", "LETÍCIA", "GRAÇA", "SANTO ANDRÉ", "SÃO LUCAS"]
NOROESTE = ["COMITECO", "CALAFATE", "SILVEIRA", "ALTO BARROCA", "MANACÁS", "BARROCA", "NOVA CINTRA", "SALGADO FILHO", "JARDIM AMÉRICA", "LUXEMBURGO", "CAIÇARA-ADELAIDE", "CARLOS PRATES", "BONSUCESSO", "CARDOSO", "CAMPO ALEGRE"]
NORTE = ["VILA CLORIS", "VILA PARIS", "CENÁCULO", "SÃO FRANCISCO", "VILA BARRAGEM SANTA LÚCIA", "RENASCENÇA", "SANTA INÊS", "PALMARES", "UNIÃO", "NOVO TUPI", "JARDIM LEBLON", "PINDORAMA", "NOVO GLÓRIA", "INDAIÁ", "JARDIM GUANABARA", "CONJUNTO PAULO VI", "JAQUELINE", "MINASCAIXA", "RIO BRANCO", "FREI LEOPOLDO"]
OESTE = ["CASTELO", "ESTORIL", "BETÂNIA", "PLANALTO", "ALÍPIO DE MELO", "CALIFÓRNIA", "SANTA CRUZ", "VENDA NOVA", "HORTO", "SERRA VERDE", "LAGOINHA LEBLON", "SÃO JOÃO BATISTA", "INCONFIDÊNCIA", "UNIVERSITÁRIO", "ETELVINA CARNEIRO", "CÉU AZUL", "SANTA BRANCA", "SANTA ROSA", "AEROPORTO", "JARAGUÁ", "PAMPULHA", "CAIÇARAS", "GRAJAÚ", "VENTOSA", "GARÇAS"]
NORDESTE = ["CINQUENTENÁRIO", "FERNÃO DIAS", "CORAÇÃO DE JESUS", "ITAPOÃ", "TREVO", "CONJUNTO CALIFÓRNIA", "IPIRANGA", "MONSENHOR MESSIAS", "SANTA MÔNICA", "PAQUETÁ", "DONA CLARA", "SERRANO", "BANDEIRANTES", "DOM BOSCO", "MANTIQUEIRA", "ESTRELA DO ORIENTE"]
BARREIRO = ["OLARIA", "SANTA HELENA", "SANTA RITA", "MILIONÁRIOS", "PONGELUPE", "DIAMANTE", "TIROL", "TEIXEIRA DIAS", "SANTA MARGARIDA", "CONJUNTO HABITACIONAL VALE DO JATOBÁ", "LINDÉIA", "PALMEIRAS", "BARREIRO", "NOVA GRANADA", "BURITIS","MILIONÁRIOS (BARREIRO)"]

zap_silver = zap_silver.withColumn("REGIÃO", when(col("BAIRRO").isin(CENTRO_SUL), "CENTRO-SUL").when(col("BAIRRO").isin(NOROESTE), "NOROESTE").when(col("BAIRRO").isin(NORTE), "NORTE").when(col("BAIRRO").isin(OESTE), "OESTE").when(col("BAIRRO").isin(NORDESTE), "NORDESTE").when(col("BAIRRO").isin(BARREIRO), "BARREIRO").otherwise("OUTROS"))

logging.info("coluna região criada com sucesso ! ")



## Fazer a contagem das regiões e confrontar com o total de registros
zap_silver.groupBy("REGIÃO").count().show()

display(zap_silver.head(100))


+----------+-----+
|    REGIÃO|count|
+----------+-----+
|CENTRO-SUL|  381|
|  NOROESTE|   37|
|     NORTE|   71|
|    OUTROS|   54|
|  BARREIRO|  276|
|     OESTE|  131|
|  NORDESTE|  100|
+----------+-----+



PRECO,CONDOMINIO,AREA,QUARTOS,BAIRRO,LATITUDE,LONGITUDE,tipo,BANHEIROS,VAGAS,IPTU,DATA_ATUALIZACAO,TIPO DE IMOVEL,TIPO_DE_IMOVEL,SEGMENTO,REGIÃO
6500,1535,680,3,CRUZEIRO,null,null,APARTMENT,3,2.0,555,2026-05-12T22:52:01.415Z,APARTAMENTO,APARTAMENTO,RESIDENCIAL,CENTRO-SUL
2500,300,98,3,MILIONÁRIOS (BARREIRO),null,null,APARTMENT,2,1.0,198,2026-05-12T22:52:01.415Z,APARTAMENTO,APARTAMENTO,RESIDENCIAL,BARREIRO
2800,0,120,3,IPÊ,null,null,HOME,2,1.0,100,2026-05-12T22:52:01.415Z,CASA,CASA,RESIDENCIAL,CENTRO-SUL
4700,512,1000,3,BURITIS,null,null,APARTMENT,3,2.0,412,2026-05-12T22:52:01.415Z,APARTAMENTO,APARTAMENTO,RESIDENCIAL,BARREIRO
4000,630,80,3,BURITIS,-19.971905,-43.965046,APARTMENT,2,2.0,2554,2026-05-12T22:52:01.415Z,APARTAMENTO,APARTAMENTO,RESIDENCIAL,BARREIRO
3500,350,720,3,SILVEIRA,null,null,APARTMENT,3,3.0,266,2026-05-12T22:52:01.415Z,APARTAMENTO,APARTAMENTO,RESIDENCIAL,NOROESTE
3300,550,90,3,ESTRELA DO ORIENTE,-19.964745,-43.984511,APARTMENT,2,2.0,290,2026-05-12T22:52:01.415Z,APARTAMENTO,APARTAMENTO,RESIDENCIAL,NORDESTE
6000,0,180,3,PAQUETÁ,null,null,HOME,2,2.0,155,2026-05-12T22:52:01.415Z,CASA,CASA,RESIDENCIAL,NORDESTE
3000,380,116,3,CAIÇARA-ADELAIDE,null,null,APARTMENT,2,2.0,190,2026-05-12T22:52:01.415Z,APARTAMENTO,APARTAMENTO,RESIDENCIAL,NOROESTE
1950,null,null,3,CONJUNTO PAULO VI,-19.83209,-43.88831,HOME,2,2.0,0,2026-05-12T22:52:01.415Z,CASA,CASA,RESIDENCIAL,NORTE


### Tratamento dos valores nulos das colunas area , vagas , iptu pela mediana 

In [0]:
### Verificação dos valores nulos nas  colunas  area 
zap_silver.filter(zap_silver.AREA.isNull()).count()
logging.info("Registros com valores nulos na coluna area: ", zap_silver.filter(zap_silver.AREA.isNull()).count())

### Tratar valores nulos pela mediana da coluna area
zap_silver = zap_silver.fillna(zap_silver.select(F.median("AREA")).collect()[0][0], subset=["AREA"])
logging.info("Registros tratados com sucesso")




### Verificação dos valores nulos nas  colunas  vagas 
zap_silver.filter(zap_silver.VAGAS.isNull()).count()
logging.info("total de registros com valores nulos na coluna vagas: ", zap_silver.filter(zap_silver.VAGAS.isNull()).count())

### Tratar valores nulos pela media da coluna vagas
zap_silver = zap_silver.fillna(zap_silver.select(F.mean("VAGAS")).collect()[0][0], subset=["VAGAS"])
logging.info("Registros tratados com sucesso")




### Verificação dos valores nulos nas  colunas  iptu
zap_silver.filter(zap_silver.IPTU.isNull()).count()
logging.info("total de registros com valores nulos na coluna iptu: ", zap_silver.filter(zap_silver.IPTU.isNull()).count())

### Tratar valores nulos pela mediana da coluna iptu
zap_silver = zap_silver.fillna(zap_silver.select(F.median("IPTU")).collect()[0][0], subset=["IPTU"])
logging.info("Registros tratados com sucesso")



# Decisão: Condomínio nulo preenchido com 0
### Motivo: Imóveis do tipo HOME legitimamente não possuem condomínio. Nulos serão preenchidos com 0 para não impactar o cálculo do CUSTO TOTAL.

In [0]:
### Verificação dos valores nulos nas  coluna condominio
zap_silver.filter(zap_silver.CONDOMINIO.isNull()).count()
logging.info("total de registros com valores nulos na coluna condominio: ", zap_silver.filter(zap_silver.CONDOMINIO.isNull()).count())

### valores nulos sera tratado com zero
zap_silver = zap_silver.fillna(0, subset=["CONDOMINIO"])
logging.info("Registros tratados com sucesso")



In [0]:
## Criar a coluna valor por metro quadrado arredondado para 02 casas decimais 

zap_silver = zap_silver.withColumn("VALOR_METRO_QUADRADO", round(col("PRECO")/col("AREA"),2))

logging.info("coluna valor por metro quadrado criada com sucesso ! ")



display(zap_silver.head(10))


PRECO,CONDOMINIO,AREA,QUARTOS,BAIRRO,LATITUDE,LONGITUDE,tipo,BANHEIROS,VAGAS,IPTU,DATA_ATUALIZACAO,TIPO DE IMOVEL,TIPO_DE_IMOVEL,SEGMENTO,REGIÃO,VALOR_METRO_QUADRADO
6500,1535,680,3,CRUZEIRO,null,null,APARTMENT,3,2.0,555,2026-05-12T22:52:01.415Z,APARTAMENTO,APARTAMENTO,RESIDENCIAL,CENTRO-SUL,9.56
2500,300,98,3,MILIONÁRIOS (BARREIRO),null,null,APARTMENT,2,1.0,198,2026-05-12T22:52:01.415Z,APARTAMENTO,APARTAMENTO,RESIDENCIAL,BARREIRO,25.51
2800,0,120,3,IPÊ,null,null,HOME,2,1.0,100,2026-05-12T22:52:01.415Z,CASA,CASA,RESIDENCIAL,CENTRO-SUL,23.33
4700,512,1000,3,BURITIS,null,null,APARTMENT,3,2.0,412,2026-05-12T22:52:01.415Z,APARTAMENTO,APARTAMENTO,RESIDENCIAL,BARREIRO,4.7
4000,630,80,3,BURITIS,-19.971905,-43.965046,APARTMENT,2,2.0,2554,2026-05-12T22:52:01.415Z,APARTAMENTO,APARTAMENTO,RESIDENCIAL,BARREIRO,50.0
3500,350,720,3,SILVEIRA,null,null,APARTMENT,3,3.0,266,2026-05-12T22:52:01.415Z,APARTAMENTO,APARTAMENTO,RESIDENCIAL,NOROESTE,4.86
3300,550,90,3,ESTRELA DO ORIENTE,-19.964745,-43.984511,APARTMENT,2,2.0,290,2026-05-12T22:52:01.415Z,APARTAMENTO,APARTAMENTO,RESIDENCIAL,NORDESTE,36.67
6000,0,180,3,PAQUETÁ,null,null,HOME,2,2.0,155,2026-05-12T22:52:01.415Z,CASA,CASA,RESIDENCIAL,NORDESTE,33.33
3000,380,116,3,CAIÇARA-ADELAIDE,null,null,APARTMENT,2,2.0,190,2026-05-12T22:52:01.415Z,APARTAMENTO,APARTAMENTO,RESIDENCIAL,NOROESTE,25.86
1950,0,100,3,CONJUNTO PAULO VI,-19.83209,-43.88831,HOME,2,2.0,0,2026-05-12T22:52:01.415Z,CASA,CASA,RESIDENCIAL,NORTE,19.5


In [0]:
## Criação da coluna custo total do imovel , arredondar o valor para 2 casas decimais

zap_silver = zap_silver.withColumn("CUSTO_TOTAL",round(col("PRECO") + col("IPTU") + col("CONDOMINIO"), 2)).orderBy(col("CUSTO_TOTAL").desc())

logging.info("coluna custo total criada com sucesso ! ")

display(zap_silver.head(10))

PRECO,CONDOMINIO,AREA,QUARTOS,BAIRRO,LATITUDE,LONGITUDE,tipo,BANHEIROS,VAGAS,IPTU,DATA_ATUALIZACAO,TIPO DE IMOVEL,TIPO_DE_IMOVEL,SEGMENTO,REGIÃO,VALOR_METRO_QUADRADO,CUSTO_TOTAL
4500000,0,100,3,SAVASSI,-19.937764,-43.932361,HOME,3,3.0,1500,2026-05-12T22:52:01.415Z,CASA,CASA,RESIDENCIAL,CENTRO-SUL,45000.0,4501500
335000,330,100,3,BURITIS,-19.966854,-43.968195,APARTMENT,2,1.0,145,2026-05-12T22:52:01.415Z,APARTAMENTO,APARTAMENTO,RESIDENCIAL,BARREIRO,3350.0,335475
21000,0,332,3,COMITECO,-19.954984,-43.925445,HOME,6,3.0,7764,2026-05-12T22:52:01.415Z,CASA,CASA,RESIDENCIAL,NOROESTE,63.25,28764
25000,2200,264,3,LOURDES,-19.929017,-43.945597,PENTHOUSE,3,3.0,1259,2026-05-12T22:52:01.415Z,COBERTURA,COBERTURA,RESIDENCIAL,CENTRO-SUL,94.7,28459
25000,2300,105,3,SAVASSI,-19.937971,-43.940789,APARTMENT,3,0.0,764,2026-05-12T22:52:01.415Z,APARTAMENTO,APARTAMENTO,RESIDENCIAL,CENTRO-SUL,238.1,28064
18000,4074,304,3,BELVEDERE,-19.971496,-43.944814,PENTHOUSE,2,4.0,1334,2026-05-12T22:52:01.415Z,COBERTURA,COBERTURA,RESIDENCIAL,CENTRO-SUL,59.21,23408
14000,0,256,3,BELVEDERE,null,null,HOME,4,4.0,7206,2026-05-12T22:52:01.415Z,CASA,CASA,RESIDENCIAL,CENTRO-SUL,54.69,21206
16500,2000,150,3,ANCHIETA,null,null,APARTMENT,4,4.0,850,2026-05-12T22:52:01.415Z,APARTAMENTO,APARTAMENTO,RESIDENCIAL,CENTRO-SUL,110.0,19350
16500,2000,150,3,ANCHIETA,-19.951501,-43.925669,APARTMENT,4,4.0,219,2026-05-12T22:52:01.415Z,APARTAMENTO,APARTAMENTO,RESIDENCIAL,CENTRO-SUL,110.0,18719
11000,0,365,3,SÃO PEDRO,null,null,COMMERCIAL_PROPERTY,2,3.0,6038,2026-05-12T22:52:01.415Z,IMOVEL COMERCIAL,IMOVEL COMERCIAL,COMERCIAL,CENTRO-SUL,30.14,17038


### Decisão: Latitude e Longitude removidas
Motivo: Alto percentual de valores nulos (~60%) inviabiliza 
análise geoespacial. Colunas removidas para simplificar o dataset.

In [0]:
## Retirar as colunas longetide e latitude, data atualizacao, tipo, tipo do imovel

zap_silver = zap_silver.drop("DATA_ATUALIZACAO")
zap_silver = zap_silver.drop("LONGITUDE","LATITUDE")
zap_silver = zap_silver.drop("tipo")
zap_silver = zap_silver.drop("DATA_ATUALIZACAO")
zap_silver = zap_silver.drop("TIPO DE IMOVEL")


display(zap_silver.head(10))



PRECO,CONDOMINIO,AREA,QUARTOS,BAIRRO,BANHEIROS,VAGAS,IPTU,TIPO_DE_IMOVEL,SEGMENTO,REGIÃO,VALOR_METRO_QUADRADO,CUSTO_TOTAL
4500000,0,100,3,SAVASSI,3,3.0,1500,CASA,RESIDENCIAL,CENTRO-SUL,45000.0,4501500
335000,330,100,3,BURITIS,2,1.0,145,APARTAMENTO,RESIDENCIAL,BARREIRO,3350.0,335475
21000,0,332,3,COMITECO,6,3.0,7764,CASA,RESIDENCIAL,NOROESTE,63.25,28764
25000,2200,264,3,LOURDES,3,3.0,1259,COBERTURA,RESIDENCIAL,CENTRO-SUL,94.7,28459
25000,2300,105,3,SAVASSI,3,0.0,764,APARTAMENTO,RESIDENCIAL,CENTRO-SUL,238.1,28064
18000,4074,304,3,BELVEDERE,2,4.0,1334,COBERTURA,RESIDENCIAL,CENTRO-SUL,59.21,23408
14000,0,256,3,BELVEDERE,4,4.0,7206,CASA,RESIDENCIAL,CENTRO-SUL,54.69,21206
16500,2000,150,3,ANCHIETA,4,4.0,850,APARTAMENTO,RESIDENCIAL,CENTRO-SUL,110.0,19350
16500,2000,150,3,ANCHIETA,4,4.0,219,APARTAMENTO,RESIDENCIAL,CENTRO-SUL,110.0,18719
11000,0,365,3,SÃO PEDRO,2,3.0,6038,IMOVEL COMERCIAL,COMERCIAL,CENTRO-SUL,30.14,17038


Nesta analise verificado que tem um imovel com um valor muito acima dos demais , possivel outleir, pode ser um erro que veio do dataset ou imovel de alta renda . 
 
Será salvo em metricas para analise na gold

In [0]:
## Verificação de valores outleirs na coluna preco
outleirs = zap_silver.filter(col("CUSTO_TOTAL") > 1000000)
display(outleirs.head(10))


PRECO,CONDOMINIO,AREA,QUARTOS,BAIRRO,BANHEIROS,VAGAS,IPTU,TIPO_DE_IMOVEL,SEGMENTO,REGIÃO,VALOR_METRO_QUADRADO,CUSTO_TOTAL
4500000,0,100,3,SAVASSI,3,3.0,1500,CASA,RESIDENCIAL,CENTRO-SUL,45000.0,4501500


In [0]:
## criar coluna data de carga

from datetime import datetime

zap_silver = zap_silver.withColumn("DATA_CARGA",lit(datetime.now()))

display(zap_silver.head(10))


PRECO,CONDOMINIO,AREA,QUARTOS,BAIRRO,BANHEIROS,VAGAS,IPTU,TIPO_DE_IMOVEL,SEGMENTO,REGIÃO,VALOR_METRO_QUADRADO,CUSTO_TOTAL,DATA_CARGA
4500000,0,100,3,SAVASSI,3,3.0,1500,CASA,RESIDENCIAL,CENTRO-SUL,45000.0,4501500,2026-05-14T22:00:26.039Z
335000,330,100,3,BURITIS,2,1.0,145,APARTAMENTO,RESIDENCIAL,BARREIRO,3350.0,335475,2026-05-14T22:00:26.039Z
21000,0,332,3,COMITECO,6,3.0,7764,CASA,RESIDENCIAL,NOROESTE,63.25,28764,2026-05-14T22:00:26.039Z
25000,2200,264,3,LOURDES,3,3.0,1259,COBERTURA,RESIDENCIAL,CENTRO-SUL,94.7,28459,2026-05-14T22:00:26.039Z
25000,2300,105,3,SAVASSI,3,0.0,764,APARTAMENTO,RESIDENCIAL,CENTRO-SUL,238.1,28064,2026-05-14T22:00:26.039Z
18000,4074,304,3,BELVEDERE,2,4.0,1334,COBERTURA,RESIDENCIAL,CENTRO-SUL,59.21,23408,2026-05-14T22:00:26.039Z
14000,0,256,3,BELVEDERE,4,4.0,7206,CASA,RESIDENCIAL,CENTRO-SUL,54.69,21206,2026-05-14T22:00:26.039Z
16500,2000,150,3,ANCHIETA,4,4.0,850,APARTAMENTO,RESIDENCIAL,CENTRO-SUL,110.0,19350,2026-05-14T22:00:26.039Z
16500,2000,150,3,ANCHIETA,4,4.0,219,APARTAMENTO,RESIDENCIAL,CENTRO-SUL,110.0,18719,2026-05-14T22:00:26.039Z
11000,0,365,3,SÃO PEDRO,2,3.0,6038,IMOVEL COMERCIAL,COMERCIAL,CENTRO-SUL,30.14,17038,2026-05-14T22:00:26.039Z


In [0]:
## Salvar os dados na tabela silver

patch_silver = "/Volumes/workspace/zap_imoveis/silver"

zap_silver.write \
  .format("delta") \
  .mode("overwrite") \
  .save(f"{patch_silver}/zap_silver")
 


logging.info("tabela delta criada com sucesso na camada silver")




In [0]:
## Salvar o outlier como  metrica da silver

outleirs.write \
  .format("delta") \
  .mode("overwrite") \
  .saveAsTable(f"{patch_silver}/outliers")

 


logging.info("metrica criada com sucesso na camada silver")

---------------------------------------------------------------------------
AnalysisException                         Traceback (most recent call last)
File <command-4695261688096443>, line 6
      1 ## Salvar o outlier como  metrica da silver
      3 outleirs.write \
      4   .format("delta") \
      5   .mode("overwrite") \
----> 6   .saveAsTable(f"{patch_silver}/outliers")
     11 logging.info("metrica criada com sucesso na camada silver")

File /databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/readwriter.py:737, in DataFrameWriter.saveAsTable(self, name, format, mode, partitionBy, **options)
    735 self._write.table_name = name
    736 self._write.table_save_method = "save_as_table"
--> 737 _, _, ei = self._spark.client.execute_command(
    738     self._write.command(self._spark.client), self._write.observations
    739 )
    740 self._callback(ei)

File /databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/client/core.py:1538, in SparkConnectClie

In [0]:
## ler os dados da silver
df_silver = spark.read.format("delta").load(f"{patch_silver}/zap_silver")

display(df_silver.head(10))

## Verificar se os dados foram gravados na silver
df_silver.count()

PRECO,CONDOMINIO,AREA,QUARTOS,BAIRRO,BANHEIROS,VAGAS,IPTU,TIPO_DE_IMOVEL,SEGMENTO,REGIÃO,VALOR_METRO_QUADRADO,CUSTO_TOTAL,DATA_CARGA
4500000,0,100,3,SAVASSI,3,3.0,1500,CASA,RESIDENCIAL,CENTRO-SUL,45000.0,4501500,2026-05-14T22:00:26.039Z
335000,330,100,3,BURITIS,2,1.0,145,APARTAMENTO,RESIDENCIAL,BARREIRO,3350.0,335475,2026-05-14T22:00:26.039Z
21000,0,332,3,COMITECO,6,3.0,7764,CASA,RESIDENCIAL,NOROESTE,63.25,28764,2026-05-14T22:00:26.039Z
25000,2200,264,3,LOURDES,3,3.0,1259,COBERTURA,RESIDENCIAL,CENTRO-SUL,94.7,28459,2026-05-14T22:00:26.039Z
25000,2300,105,3,SAVASSI,3,0.0,764,APARTAMENTO,RESIDENCIAL,CENTRO-SUL,238.1,28064,2026-05-14T22:00:26.039Z
18000,4074,304,3,BELVEDERE,2,4.0,1334,COBERTURA,RESIDENCIAL,CENTRO-SUL,59.21,23408,2026-05-14T22:00:26.039Z
14000,0,256,3,BELVEDERE,4,4.0,7206,CASA,RESIDENCIAL,CENTRO-SUL,54.69,21206,2026-05-14T22:00:26.039Z
16500,2000,150,3,ANCHIETA,4,4.0,850,APARTAMENTO,RESIDENCIAL,CENTRO-SUL,110.0,19350,2026-05-14T22:00:26.039Z
16500,2000,150,3,ANCHIETA,4,4.0,219,APARTAMENTO,RESIDENCIAL,CENTRO-SUL,110.0,18719,2026-05-14T22:00:26.039Z
11000,0,365,3,SÃO PEDRO,2,3.0,6038,IMOVEL COMERCIAL,COMERCIAL,CENTRO-SUL,30.14,17038,2026-05-14T22:00:26.039Z


1050